### Starter pack for the Brain age prediction EEG data set (Helsinki 1K-Kids) : Data loader

**Last updated:** 31.10.2026

This is a basic dataloader for the pre-processed EEG dataset.

In [ ]:
import os
import torch
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, file_path, segments, segment_duration, sampling_rate, cache_size=None):
        self.file_path = file_path
        self.segments = segments
        self.segment_duration = 1.0*segment_duration
        self.sampling_rate = sampling_rate
        if cache_size is None:
            self.max_cache_size = 0
        elif cache_size == 'inf':
            self.max_cache_size = 10E9 # Number of elements (2 bytes per element)
        else:
            self.max_cache_size = cache_size # Max number of samples stored in the cache on each worker
        self.cache_size = 0
        self.file_cache = {}

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        torch_file = self.segments[idx][4]
        istart = int(self.segments[idx][0]*self.sampling_rate)
        label = self.segments[idx][3]
        n_samples = int(self.sampling_rate*self.segment_duration)

        # Caching - Store EEG in a dictionary for fast lookup
        if torch_file in self.file_cache:
            data = self.file_cache[torch_file]
        else:
            data = torch.load(os.path.join(self.file_path, 'pre_processed/' + torch_file.replace('.edf','.pt')))
            if self.cache_size + torch.numel(data) < self.max_cache_size:
                self.file_cache[torch_file] = data
                self.cache_size += torch.numel(data)
        assert(data.shape[1] >= n_samples)

        # Starting point of segment is specified by b_start
        # It is zero currently which corresponds to the start of the recording
        # NOTE: If you want to use random cropping you can implement it by modifying b_start
        b_start = self.segments[idx][2]
        istart = int(b_start*self.sampling_rate)
        data1 = data[:, istart:(istart + n_samples)] # Copy data prior to type conversion/augmentation
        assert(data1.shape[1] == n_samples) # slicing out of bounds in pytorch is silent, therefore need to check
        if data1.dtype != torch.float32:
            data1 = data1.to(torch.float32) # Pytorch operates in float32 by default

        # TODO: Add augmentation code here if you decide to use augmentation
        return data1, label, idx

    def clear_cache():
        self.file_cache.clear()
        self.cache_size = 0

In [ ]:
import os
from pathlib import Path
import torch

RESAMPLING_RATE = 64
SEGMENT_DURATION = 8 # TODO: Decide on appropriate block size for your model

BATCH_SIZE = 64 # TODO: Modify as needed

path = Path('data')
pre_processed_path = path / Path('pre_processed')

train_segments = torch.load(path / 'segments_h1k_train.pt')
val_segments = torch.load(path / 'segments_h1k_val.pt')
test_segments = torch.load(path / 'segments_h1k_test.pt')

In [ ]:
from torch.utils.data import DataLoader

train_dataset = EEGDataset(path, train_segments,
                           segment_duration=SEGMENT_DURATION,
                           sampling_rate=RESAMPLING_RATE,
                           cache_size=2E9)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers = 0)

it = iter(train_loader)
data, label, _ = next(it)

data.shape, len(label)